In [0]:
CREATE OR REPLACE TABLE gdp_df1 (country STRING, year INT, gdp DOUBLE);

INSERT INTO gdp_df1
  VALUES ('USA', 2018, 20544.34), ('USA', 2019, 21427.70), ('China', 2018, 13894.04);

CREATE OR REPLACE TABLE gdp_df2 (country STRING, year INT, gdp DOUBLE);

INSERT INTO gdp_df2
  VALUES ('China', 2019, 14402.72), ('India', 2018, 2713.61), ('India', 2019, 2868.93);

with gdp_union_df as (
  select
    *
  from
    gdp_df1
  union all
  select
    *
  from
    gdp_df2
),
gdp_df as (
  select
    country,
    year,
    gdp as current_gdp,
    lag(gdp) over (partition by country order by year) as previous_gdp
  from
    gdp_union_df
),
gdp_growth_df as (
  select
    country,
    year,
    round((current_gdp - previous_gdp) * 100 / (previous_gdp), 2) as gpd_growth_rate
  from
    gdp_df
)
select
  country,
  year,
  COALESCE(CAST(gpd_growth_rate AS STRING), '') as gpd_growth_rate
from
  gdp_growth_df